In [1]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
from transformers import pipeline
import warnings
warnings.filterwarnings("ignore")

# Connect to BigQuery
client = bigquery.Client(project="project-a13303d6-d497-41fa-ab8")
print("✅ Connected to BigQuery!")

# Load sentiment analysis pipeline
# This uses a pretrained financial sentiment model
# No training needed — it already knows finance language
print("\nLoading sentiment model...")
sentiment_model = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    truncation=True
)
print("✅ FinBERT sentiment model loaded!")

✅ Connected to BigQuery!

Loading sentiment model...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✅ FinBERT sentiment model loaded!


In [2]:
# ============================================================
# TEST SENTIMENT MODEL
# ============================================================

test_headlines = [
    "Oil prices surge to highest level in 3 months amid OPEC cuts",
    "Bitcoin crashes 15% as regulators announce crackdown",
    "Gold steady as investors await Federal Reserve decision",
    "S&P 500 hits record high driven by strong earnings",
    "Economic recession fears grow as unemployment rises",
    "Saudi Arabia increases oil production sending prices lower"
]

print("=== FINBERT SENTIMENT TEST ===\n")
for headline in test_headlines:
    result = sentiment_model(headline)[0]
    label = result["label"]
    score = result["score"]
    
    emoji = "🟢" if label == "positive" else "🔴" if label == "negative" else "🟡"
    print(f"{emoji} {label.upper()} ({score:.2f}) — {headline}")

=== FINBERT SENTIMENT TEST ===

🟢 POSITIVE (0.90) — Oil prices surge to highest level in 3 months amid OPEC cuts
🔴 NEGATIVE (0.90) — Bitcoin crashes 15% as regulators announce crackdown
🟢 POSITIVE (0.42) — Gold steady as investors await Federal Reserve decision
🟢 POSITIVE (0.92) — S&P 500 hits record high driven by strong earnings
🔴 NEGATIVE (0.95) — Economic recession fears grow as unemployment rises
🟢 POSITIVE (0.88) — Saudi Arabia increases oil production sending prices lower


In [4]:
import requests
from datetime import datetime, timedelta

# ============================================================
# PULL NEWS FROM NEWSAPI
# ============================================================

API_KEY = "ec98084f051047cba5af54bc750c7e30"

# Search terms for each asset
search_terms = {
    "Oil":     "crude oil price OPEC",
    "Gold":    "gold price market",
    "SP500":   "S&P 500 stock market",
    "Bitcoin": "bitcoin crypto price"
}

all_headlines = []

for asset, query in search_terms.items():
    print(f"Pulling news for {asset}...")
    
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": query,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": 100,
        "from": (datetime.now() - timedelta(days=28)).strftime("%Y-%m-%d"),
        "apiKey": API_KEY
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if data["status"] == "ok":
        articles = data["articles"]
        for article in articles:
            if article["title"] and article["title"] != "[Removed]":
                all_headlines.append({
                    "date": article["publishedAt"][:10],
                    "asset": asset,
                    "headline": article["title"],
                    "source": article["source"]["name"]
                })
        print(f"✅ {asset}: {len(articles)} headlines")
    else:
        print(f"❌ {asset}: {data.get('message', 'Error')}")

news_df = pd.DataFrame(all_headlines)
print(f"\nTotal headlines: {len(news_df)}")
print(news_df.head())

Pulling news for Oil...
✅ Oil: 94 headlines
Pulling news for Gold...
✅ Gold: 94 headlines
Pulling news for SP500...
✅ SP500: 95 headlines
Pulling news for Bitcoin...
✅ Bitcoin: 100 headlines

Total headlines: 383
         date asset                                           headline  \
0  2026-03-11   Oil  Saudi Arabia Announces Unilateral Cuts to Oil ...   
1  2026-03-11   Oil   Oil crisis: Is world better placed than in 1973?   
2  2026-03-10   Oil  The global oil crisis is even worse than it looks   
3  2026-03-10   Oil          Current price of oil as of March 10, 2026   
4  2026-03-10   Oil   Oil crisis: Is world better placed than in 1973?   

                source  
0      Naturalnews.com  
1  Hurriyet Daily News  
2                  Vox  
3              Fortune  
4      Digital Journal  


In [5]:
# ============================================================
# RUN SENTIMENT ANALYSIS ON ALL HEADLINES
# ============================================================

print("Running sentiment analysis on 383 headlines...")
print("This will take 2-3 minutes...\n")

sentiments = []
for i, headline in enumerate(news_df["headline"]):
    result = sentiment_model(headline[:512])[0]  # truncate to 512 chars
    sentiments.append({
        "label": result["label"],
        "score": round(result["score"], 4)
    })
    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/383 headlines...")

# Add sentiment to dataframe
news_df["sentiment_label"] = [s["label"] for s in sentiments]
news_df["sentiment_score"] = [s["score"] for s in sentiments]

# Convert label to numeric score
# positive = 1, neutral = 0, negative = -1
sentiment_map = {"positive": 1, "neutral": 0, "negative": -1}
news_df["sentiment_numeric"] = news_df["sentiment_label"].map(sentiment_map)

print("\n✅ Sentiment analysis complete!")
print("\n=== SENTIMENT DISTRIBUTION BY ASSET ===")
print(news_df.groupby(["asset", "sentiment_label"]).size().unstack(fill_value=0))

Running sentiment analysis on 383 headlines...
This will take 2-3 minutes...

Processed 50/383 headlines...
Processed 100/383 headlines...
Processed 150/383 headlines...
Processed 200/383 headlines...
Processed 250/383 headlines...
Processed 300/383 headlines...
Processed 350/383 headlines...

✅ Sentiment analysis complete!

=== SENTIMENT DISTRIBUTION BY ASSET ===
sentiment_label  negative  neutral  positive
asset                                       
Bitcoin                25       45        30
Gold                   15       54        25
Oil                    34       37        23
SP500                  45       30        20


In [6]:
# ============================================================
# WRITE NEWS & SENTIMENT TO BIGQUERY
# ============================================================

table_id = "project-a13303d6-d497-41fa-ab8.market_intelligence.news_sentiment"

job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("date", "DATE"),
        bigquery.SchemaField("asset", "STRING"),
        bigquery.SchemaField("headline", "STRING"),
        bigquery.SchemaField("source", "STRING"),
        bigquery.SchemaField("sentiment_label", "STRING"),
        bigquery.SchemaField("sentiment_score", "FLOAT"),
        bigquery.SchemaField("sentiment_numeric", "INTEGER"),
    ],
    write_disposition="WRITE_TRUNCATE"
)

news_df["date"] = pd.to_datetime(news_df["date"]).dt.date

job = client.load_table_from_dataframe(news_df, table_id, job_config=job_config)
job.result()

table = client.get_table(table_id)
print(f"✅ News sentiment table created!")
print(f"Rows in BigQuery: {table.num_rows}")

# Query to verify
query = """
    SELECT 
        asset,
        COUNT(*) as total_headlines,
        ROUND(AVG(sentiment_numeric), 3) as avg_sentiment,
        COUNTIF(sentiment_label = 'positive') as positive,
        COUNTIF(sentiment_label = 'neutral') as neutral,
        COUNTIF(sentiment_label = 'negative') as negative
    FROM `project-a13303d6-d497-41fa-ab8.market_intelligence.news_sentiment`
    GROUP BY asset
    ORDER BY avg_sentiment DESC
"""

results = client.query(query).to_dataframe()
print("\n=== SENTIMENT SUMMARY IN BIGQUERY ===")
print(results.to_string(index=False))

✅ News sentiment table created!
Rows in BigQuery: 383

=== SENTIMENT SUMMARY IN BIGQUERY ===
  asset  total_headlines  avg_sentiment  positive  neutral  negative
   Gold               94          0.106        25       54        15
Bitcoin              100          0.050        30       45        25
    Oil               94         -0.117        23       37        34
  SP500               95         -0.263        20       30        45
